# Ajustes, Filtros e Efeitos

Nesse notebook vamos explorar alguns recursos para realizar ajustes fotométricos, aplicar filtros e efeitos nas imagens.

In [ ]:
if 'google.colab' in str(get_python()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python>=5.0.0
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
else:
    pass

## Carregando bibliotecas

Nesse notebook, vamos usar as mesmas bibliotecas que trabalhamos nos notebooks anteriores.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline  
from IPython.display import Image

## 1. Ajustes Fotométricos

Ajustes fotométricos são transformações que modificam as características de iluminação, intensidade e cor de uma imagem, preservando sua estrutura geométrica e os objetos nela contidos.

### 1.1. Ajuste de Brilho e Contraste

A função `convertScaleAbs()` aplica uma transformação linear aos pixels da imagem de entrada `src`, multiplicando cada valor por um fator de escala (`alpha`) e adicionando um deslocamento (`beta`). Em seguida, calcula o valor absoluto do resultado e converte os pixels para o tipo `uint8`. <br>

---

### <font color="green">Sintaxe da Função</font>
```python
dst = cv2.convertScaleAbs(src[, dst[, alpha[, beta]]])
```

`dst`: imagem de saída. Ela terá o mesmo tamanho da imagem de entrada (`src`) e seus pixels serão convertidos para o tipo `uint8` após a aplicação da transformação linear e da operação de valor absoluto.

A função possui **1 argumento obrigatório**:

1. `src`: imagem de entrada.

Os argumentos opcionais mais utilizados incluem:

1. `alpha`: fator de escala aplicado aos valores dos pixels. O valor padrão é `1.0`. Controla o constraste da imagem.
2. `beta`: deslocamento (offset) somado aos valores dos pixels após a multiplicação por `alpha`. O valor padrão é `0`. Controla o brilho da imagem.


Essa função é frequentemente utilizada para realizar ajustes de brilho e contraste de forma simples e eficiente. Valores de `alpha` maiores que `1.0` aumentam o contraste da imagem, enquanto valores menores que `1.0` o reduzem. Já valores positivos de `beta` tornam a imagem mais clara, enquanto valores negativos a deixam mais escura.

In [ ]:
img = cv2.imread('imagens/puc-minas.jpg', cv2.IMREAD_COLOR)

# 3 sugestões de ajuste
scaleabs_img_1 = cv2.convertScaleAbs(img, alpha=2.0, beta=0.0)
scaleabs_img_2 = cv2.convertScaleAbs(img, alpha=0.3, beta=0.0)
scaleabs_img_3 = cv2.convertScaleAbs(img, alpha=1.0, beta=50.0)

# Apresenta imagens
plt.figure(figsize = [18, 5])
plt.subplot(141); plt.imshow(scaleabs_img_1[:, :, ::-1]) 
plt.title('Alto Contraste')
plt.subplot(142); plt.imshow(scaleabs_img_2[:, :, ::-1])
plt.title('Baixo Contraste')
plt.subplot(143); plt.imshow(scaleabs_img_3[:, :, ::-1]) 
plt.title('Somente Brilho')
plt.subplot(144); plt.imshow(img[:, :, ::-1])            
plt.title('Original');

### 1.2. Equalização de Histograma

A função `equalizeHist()` realiza a equalização do histograma, que é um método não-linear de melhorar o contraste de uma imagem.

---

### <font color="green">Sintaxe da Função</font>
```python
	dst = cv2.equalizeHist(src[, dst])
```

**Parâmetros:**

- `src`: Imagem com apenas um canal (normalmente em escala de cinza).
- `dst`: Imagem de destino com as mesmas dimensões da imagem de origem.

#### 1.2.1. Imagens na Escala de Cinza

In [ ]:
# Lê imagem como escala de cinza
img = cv2.imread('imagens/parrot.jpg', cv2.IMREAD_GRAYSCALE)

# Equaliza histograma
img_eq = cv2.equalizeHist(img)

# Apresenta imagens com seus respectivos histogramas
plt.figure(figsize=(15, 10))
plt.subplot(221); plt.imshow(img, cmap='gray'); plt.title('Original')
plt.subplot(222); plt.hist(img.ravel(), 256); plt.xlim([0, 256]); plt.title('Histograma Original')
plt.subplot(223); plt.imshow(img_eq, cmap='gray'); plt.title('Imagem Equalizada')
plt.subplot(224); plt.hist(img_eq.ravel(), 256); plt.xlim([0, 256]); plt.title('Histograma Equalizado');

#### 1.2.2. Imagens Coloridas

Vamos mostrar duas formas de equalização:

1. Equalizando todos os canais de cor
2. Equalizando apenas o canal de luminosidade (espaço de cor HSV)

In [ ]:
# Lê imagem como escala de cinza
img = cv2.imread('imagens/parrot.jpg', cv2.IMREAD_COLOR)

# Método 1: equalizando histograma de todos os canais
img_eq_1 = np.zeros_like(img)
for i in range(0, 3):
    img_eq_1[:, :, i] = cv2.equalizeHist(img[:, :, i])

# Método 2: equalizando canal V do espaço de cor HSV
img_eq_2 = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
img_eq_2[:, :, 2] = cv2.equalizeHist(img_eq_2[:, :, 2])
img_eq_2 = cv2.cvtColor(img_eq_2, cv2.COLOR_HSV2BGR)

# Apresenta imagens
plt.figure(figsize=(18, 6))
plt.subplot(131); plt.axis('off'); plt.imshow(img_eq_1[:, :, ::-1]); plt.title('Método 1 (errado)')
plt.subplot(132); plt.axis('off'); plt.imshow(img_eq_2[:, :, ::-1]); plt.title('Método 2 (correto)')
plt.subplot(133); plt.axis('off'); plt.imshow(img[:, :, ::-1]); plt.title('Original');

## 2. Filtros e Efeitos

Filtros e efeitos são técnicas utilizadas para modificar a aparência visual de uma imagem por meio do processamento dos seus pixels, podendo suavizar ruídos, realçar detalhes, detectar características ou produzir efeitos estéticos e artísticos.

### 2.1. Blur

A função **`GaussianBlur()`** aplica um **filtro Gaussiano de suavização (blur)** à imagem. Esse filtro reduz ruídos e detalhes finos, produzindo uma imagem mais suave. Trata-se de um dos filtros mais utilizados em visão computacional, especialmente como etapa de pré-processamento para detecção de bordas, segmentação e extração de características.

---

### <font style="color:rgb(8,133,37)">Sintaxe da Função</font>
```python
dst = cv2.GaussianBlur(src, ksize, sigmaX[, dst[, sigmaY[, borderType]]])
```

`dst`: imagem de saída com o mesmo tamanho e o mesmo tipo de dados da imagem de entrada (`src`).

A função possui **3 argumentos de entrada obrigatórios**:

1. `src`: imagem de entrada. A imagem pode possuir qualquer número de canais, que são processados independentemente. 
2. `ksize`: tamanho do kernel Gaussiano. Os valores `ksize.width` e `ksize.height` podem ser diferentes, mas ambos devem ser positivos e ímpares. Alternativamente, podem ser definidos como zero, caso em que serão calculados automaticamente a partir do valor de sigma.
3. `sigmaX`: desvio padrão da função Gaussiana na direção **X**.

Os argumentos opcionais incluem:

* `sigmaY`: desvio padrão da função Gaussiana na direção **Y**. Se `sigmaY` for igual a zero, ele assumirá o mesmo valor de `sigmaX`. Se ambos (`sigmaX` e `sigmaY`) forem zero, seus valores serão calculados automaticamente a partir de `ksize.width` e `ksize.height`, respectivamente. Para garantir controle total sobre o resultado, independentemente de possíveis mudanças futuras na implementação da função, recomenda-se especificar explicitamente **`ksize`**, **`sigmaX`** e **`sigmaY`**.
* `borderType`: método utilizado para extrapolar os valores dos pixels nas bordas da imagem durante a aplicação do filtro.

O valor de **sigma** pode ser estimado a partir do tamanho do kernel utilizando a seguinte fórmula:

$$
\sigma = 0.3 \times \left(\frac{ksize - 1}{2} - 1\right) + 0.8
$$

In [ ]:
img = cv2.imread('imagens/puc-minas.jpg', cv2.IMREAD_COLOR)

# Aplica gaussian blur
gaussian_blur1 = cv2.GaussianBlur(img, (5, 5), 0, 0)
gaussian_blur2 = cv2.GaussianBlur(img, (11, 11), 0, 0)

# Apresenta
plt.figure(figsize=(18, 10))
plt.subplot(131); plt.axis('off'); plt.imshow(img[:, :, ::-1]); plt.title('Original')
plt.subplot(132); plt.axis('off'); plt.imshow(gaussian_blur1[:, :, ::-1]); plt.title('Blur 5x5 kernel')
plt.subplot(133); plt.axis('off'); plt.imshow(gaussian_blur2[:, :, ::-1]); plt.title('Blur 11x11 kernel');

### 2.2. Nitidez (Sharpen)

A função **`filter2D()`** é uma das funções mais versáteis do OpenCV para filtragem de imagens, pois permite aplicar **qualquer kernel personalizado**. Com ela, é possível implementar filtros de suavização, realce, detecção de bordas, nitidez (*sharpening*), embossing e muitos outros tipos de processamento espacial apenas definindo diferentes matrizes de kernel.

---

### <font style="color:rgb(8,133,37)">Sintaxe da Função</font>
```python
dst = cv2.filter2D(src, ddepth, kernel[, dst[, anchor[, delta[, borderType]]]])
```

`dst`: imagem de saída com o mesmo tamanho e o mesmo número de canais da imagem de entrada (`src`).

A função possui **3 argumentos de entrada obrigatórios**:

1. `src`: imagem de entrada.
2. `ddepth`: profundidade de dados desejada para a imagem de saída.
3. `kernel`: kernel de convolução (ou, mais precisamente, de correlação), representado por uma matriz de ponto flutuante com um único canal. Caso seja necessário aplicar filtros diferentes a canais distintos, recomenda-se separar a imagem em planos de cor individuais utilizando a função `split()` e processar cada canal separadamente.

Os argumentos opcionais incluem:

1. `anchor`: ponto de ancoragem do kernel, que indica a posição relativa do pixel filtrado dentro do kernel. A âncora deve estar localizada dentro do kernel. O valor padrão **(-1, -1)** indica que a âncora está posicionada no centro do kernel.
2. `delta`: valor opcional adicionado aos pixels filtrados antes de armazenar o resultado em `dst`.
3. `borderType`: método utilizado para tratar os pixels das bordas da imagem durante a aplicação do filtro.

**Observação:** Os parâmetros opcionais acima raramente são alterados em relação aos seus valores padrão.

In [ ]:
img = cv2.imread('imagens/puc-minas.jpg', cv2.IMREAD_COLOR)

# Kernel para nitidez mais comum
kernel1 = np.array([[ 0, -1,  0],
                    [-1,  5, -1],
                    [ 0, -1,  0]])

# Kernel para nitidez mais extrema
kernel2 = np.array([[0,  -4,  0],
                   [-4,  17, -4],
                   [ 0,  -4,  0]])

# Aplica nitidez
image_sharp1 = cv2.filter2D(img, ddepth = -1, kernel = kernel1)
image_sharp2 = cv2.filter2D(img, ddepth = -1, kernel = kernel2)

# Apresenta
plt.figure(figsize=(18, 10))
plt.subplot(131); plt.axis('off'); plt.imshow(img[:, :, ::-1]); plt.title('Original')
plt.subplot(132); plt.axis('off'); plt.imshow(image_sharp1[:, :, ::-1]); plt.title('Nitidez Comum')
plt.subplot(133); plt.axis('off'); plt.imshow(image_sharp2[:, :, ::-1]); plt.title('Nitidez Exagerada');

### 2.3. CLAHE

A função `createCLAHE()` cria um objeto CLAHE (*Contrast Limited Adaptive Histogram Equalization*), utilizado para melhorar o contraste da imagem de forma adaptativa. Diferentemente da equalização de histograma convencional, o CLAHE divide a imagem em pequenas regiões (*tiles*) e aplica a equalização localmente, limitando a amplificação excessiva do contraste para reduzir o aparecimento de ruído.

O CLAHE é especialmente útil para imagens com iluminação não uniforme, como:

* imagens médicas (radiografias, tomografias);
* documentos digitalizados;
* imagens de vigilância;
* fotografias com partes muito claras e muito escuras.

---

### <font color="green">Sintaxe da Função</font>
```python
clahe = cv2.createCLAHE([clipLimit[, tileGridSize]])
```

`dst`: imagem de saída após a aplicação do CLAHE. Ela possui o mesmo tamanho da imagem de entrada (`src`) e normalmente apresenta melhor contraste local.

A função possui **0 argumentos obrigatórios**.

Os argumentos opcionais mais utilizados incluem:

1. `clipLimit`: limite para o contraste local. Controla a amplificação máxima do histograma em cada região da imagem. O valor padrão é `40.0`.
2. `tileGridSize`: número de regiões (*tiles*) em que a imagem será dividida, especificado como uma tupla `(colunas, linhas)`. O valor padrão é `(8, 8)`.

Após criar o objeto CLAHE, a transformação é aplicada utilizando:

```python
dst = clahe.apply(src)
```

onde:

* `src`: imagem de entrada, normalmente em escala de cinza;
* `dst`: imagem resultante com contraste aprimorado;
* o contraste é ajustado independentemente em cada região (*tile*);
* os resultados das regiões vizinhas são combinados suavemente para evitar descontinuidades visíveis.

O parâmetro `clipLimit` é particularmente importante:

* Valores menores limitam mais a amplificação do contraste, reduzindo também a amplificação do ruído.
* Valores maiores aumentam o contraste local, mas podem tornar o ruído mais evidente.

Já o parâmetro `tileGridSize` controla o tamanho das regiões analisadas:

* Regiões menores (por exemplo, `(4,4)` ou `(8,8)`) produzem ajustes mais localizados.
* Regiões maiores se aproximam do comportamento da equalização de histograma tradicional.

#### <font color="red">Atenção</font>

Em imagens coloridas, normalmente o CLAHE não é aplicado diretamente nos três canais RGB. Uma abordagem comum consiste em converter a imagem para o espaço de cores **HSV**, aplicar o CLAHE apenas no canal de valor (**V**) e depois reconstruir a imagem colorida. Isso evita alterações indesejadas nas cores originais.


In [ ]:
# Criando objeto CLAHE
clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

# Usando CLAHE em imagem na escala de cinza
img1 = cv2.imread('imagens/lagoa-da-pampulha.webp', cv2.IMREAD_GRAYSCALE)
img_clahe1 = clahe.apply(img1)

# Usando CLAHE em imagem colorida
img2 = cv2.imread('imagens/puc-minas.jpg', cv2.IMREAD_COLOR)
img_clahe2 = cv2.cvtColor(img2, cv2.COLOR_BGR2HSV)
img_clahe2[:, :, 2] = clahe.apply(img_clahe2[:, :, 2])
img_clahe2 = cv2.cvtColor(img_clahe2, cv2.COLOR_HSV2BGR)

# Apresenta imagens
plt.figure(figsize=(15, 10))
plt.subplot(221); plt.axis('off'); plt.imshow(img1, cmap='gray'); plt.title('Original')
plt.subplot(222); plt.axis('off'); plt.imshow(img_clahe1, cmap='gray'); plt.title('CLAHE')
plt.subplot(223); plt.axis('off'); plt.imshow(img2[:, :, ::-1]); plt.title('Original')
plt.subplot(224); plt.axis('off'); plt.imshow(img_clahe2[:, :, ::-1]); plt.title('CLAHE');